# Tool-Router Ladder: log-prob run for the escalation check (V1-18)

The fine-tuned 1.5B at catalog 80, recording each token's log-prob. About 75 minutes on a T4.

**Run it in the background, not interactively:** Save Version → **Save & Run All (Commit)**.
An interactive session dies when its tab sits idle; a commit run does not need the tab.

**Settings before committing:** Accelerator GPU T4, Internet on, the tool-router dataset
attached, secret `HF_TOKEN` ticked.

The last cell pushes the output to the Hub. A commit run also keeps `/kaggle/working/out`
as the version's output, so the file survives even if the push fails.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
!pip install -q vllm

# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)

In [ ]:
!python src/generate.py --config configs/qwen15b.yaml --adapter "$HF_USER/tool-router-qwen15b-sft" --prompts "$DATA/prompts-c80.jsonl" --out "$OUT/completions-qwen15b-sft-c80-logprobs.jsonl" --logprobs

In [ ]:
from huggingface_hub import HfApi

REPO = f"{HF_USER}/tool-router-results"
api = HfApi()
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, path_in_repo="completions", repo_id=REPO, repo_type="dataset")
print("uploaded:", sorted(os.listdir(OUT)))